In [5]:
#1. Carga limpia y preparación para el futuro
import pandas as pd
import numpy as np

# Cargar el dataset limpio
df = pd.read_csv("../01.data/processed/telco_churn_clean.csv")

# Solución al Warning: Forzar tipos explícitos para asegurar compatibilidad con Pandas 2.0 / 3.0
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].astype("string")

# Resguardamos el customerID como índice para no perder la trazabilidad ni codificarlo erróneamente
df = df.set_index("customerID")


/tmp/ipykernel_2898/1496053963.py:9: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


In [6]:
#2. Ingeniería de Características Avanzada (Aporta valor al modelo)
# 1. Ratio de cargos: Qué tanto peso tiene el cargo mensual sobre el total acumulado
df["Charges_Ratio"] = df["MonthlyCharges"] / (df["TotalCharges"] + 1e-5)

# 2. Variable combinada de riesgo: Clientes con contrato mes a mes y sin soporte técnico online
df["High_Risk_Contract"] = ((df["Contract"] == "Month-to-month") & (df["TechSupport"] == "No")).astype(int)

# 3. Segmentación de Antigüedad (Tenure) en rangos de ciclo de vida del cliente
df["Tenure_Group"] = pd.cut(df["tenure"], 
                            bins=[0, 12, 24, 48, 72, np.inf], 
                            labels=["0-1 Year", "1-2 Years", "2-4 Years", "4-6 Years", "6+ Years"]).astype("string")

print(f"Columnas después de la ingeniería de características: {df.shape[1]}")


Columnas después de la ingeniería de características: 23


In [7]:
#3. Codificación y mapeo inteligente
# Convertir la variable objetivo a binaria
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0}).astype(int)

# Seleccionar solo las columnas categóricas de texto excluyendo Churn
cat_cols_to_encode = df.select_dtypes(include=["string"]).columns.tolist()
if "Churn" in cat_cols_to_encode: cat_cols_to_encode.remove("Churn")

# Aplicar One-Hot Encoding seguro evitando la explosión dimensional
df_encoded = pd.get_dummies(df, columns=cat_cols_to_encode, drop_first=True, dtype=int)

print("--- RESULTADO FINAL ---")
print(f"Dimensiones finales del dataset: {df_encoded.shape}") 
# Deberías ver alrededor de ~35 a 40 columnas en lugar de 7,062.


--- RESULTADO FINAL ---
Dimensiones finales del dataset: (7032, 36)


In [8]:
#4. Almacenamiento del Dataset de Producción
# Guardar dataset final optimizado para entrenamiento
df_encoded.to_csv("../01.data/processed/telco_churn_model_ready.csv", index=True)
print("✅ Dataset optimizado y listo para modelado guardado exitosamente.")


✅ Dataset optimizado y listo para modelado guardado exitosamente.
